In [5]:
import numpy as np
import pandas as pd
import joblib
import os

BASE = r'C:\Users\Anushka Thite\Desktop\smart-loan-system'
RAW_PATH = os.path.join(BASE, 'data', 'raw', 'Loan Approval Prediction Dataset.csv')
MODELS_DIR = os.path.join(BASE, 'models')

stack = joblib.load(os.path.join(MODELS_DIR, 'stacking.pkl'))
preprocessor = joblib.load(os.path.join(MODELS_DIR, 'preprocessor.pkl'))

df_raw = pd.read_csv(RAW_PATH).drop(columns=['Application_ID'])


def build_features(base_dict, **overrides):
    d = base_dict.copy()
    d.update(overrides)
    d['EMI_to_Income']   = d['Monthly_EMI'] / (d['Annual_Income'] / 12 + 1)
    d['Loan_to_Income']  = d['Loan_Amount'] / (d['Annual_Income'] + 1)
    d['Savings_to_Loan'] = d['Savings'] / (d['Loan_Amount'] + 1)
    d['Net_Worth']       = d['Savings'] + d['Investments'] - d['Existing_Loan_Amount']
    d['Has_Collateral']  = 1 if d['Collateral'] == 'Yes' else 0
    d.pop('Loan_Status', None)
    return d


def predict_probability(feature_dict):
    df_input = pd.DataFrame([feature_dict])
    X_proc = preprocessor.transform(df_input)
    return stack.predict_proba(X_proc)[0, 1]


# Scan rejected applicants
print("Scanning rejected applicants for a TRUE borderline case...")
rejected = df_raw[df_raw['Loan_Status'] == 'Rejected'].head(500).copy()
probs = [predict_probability(build_features(row)) for row in rejected.to_dict('records')]
rejected['approval_prob'] = probs

# Filter to those the model ALSO rejects, then pick closest to 0.5
true_rejections = rejected[rejected['approval_prob'] < 0.5]
print(f"Found {len(true_rejections)} model-confirmed rejections out of 500 scanned.")

borderline = true_rejections.sort_values('approval_prob', ascending=False).iloc[0]
sample = borderline.drop('approval_prob').to_dict()
sample_features = build_features(sample)

print("=" * 60)
print("BORDERLINE REJECTED APPLICANT (model-confirmed rejection)")
print("=" * 60)
print(f"  Monthly EMI   : ₹{sample['Monthly_EMI']:,}")
print(f"  Loan Amount   : ₹{sample['Loan_Amount']:,}")
print(f"  Loan Tenure   : {sample['Loan_Tenure']} years")
print(f"  Interest Rate : {sample['Interest_Rate']}%")
print(f"  Credit Score  : {sample['Credit_Score']}")
print(f"  DTI           : {sample['Debt_to_Income']}")
print()

original_prob = predict_probability(sample_features)
print(f"Original approval probability: {original_prob:.4f}")
print(f"Original decision: REJECTED")
print()

print("=" * 60)
print("GENERATING RECOMMENDATIONS")
print("=" * 60)

recommendations = []
for amt_factor in [0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.6, 0.5, 0.4, 0.3]:
    new_amt = int(sample['Loan_Amount'] * amt_factor)
    for tenure_add in [0, 2, 5, 10, 15, 20]:
        new_tenure = sample['Loan_Tenure'] + tenure_add
        candidate = build_features(sample, Loan_Amount=new_amt, Loan_Tenure=new_tenure)
        prob = predict_probability(candidate)
        recommendations.append({
            'Loan_Amount': new_amt,
            'Loan_Tenure': new_tenure,
            'Approval_Prob': round(prob, 4),
            'Amt_Change_%': round((amt_factor - 1) * 100, 1),
            'Tenure_Change_Yr': tenure_add,
            'Approved': prob >= 0.5
        })

rec_df = pd.DataFrame(recommendations).sort_values('Approval_Prob', ascending=False)
print("\nTop 10 configurations:")
print(rec_df.head(10).to_string(index=False))

flipped = rec_df[rec_df['Approved'] == True]
print(f"\n{'=' * 60}")
print("CONFIGURATIONS THAT FLIP REJECTION → APPROVAL")
print("=" * 60)

if len(flipped) > 0:
    print(flipped.head(5).to_string(index=False))
else:
    print("No single configuration flips the decision.")
    print("This applicant is too risky. Try in combination:")
    print("  - Add co-applicant income")
    print("  - Add collateral")
    print("  - Lower interest rate (rate reduction)")
    print("  - Improve credit score (long-term)")

rec_df.to_csv(os.path.join(MODELS_DIR, 'recommendations_borderline.csv'), index=False)
print(f"\n✅ Saved to models/recommendations_borderline.csv")

Scanning rejected applicants for a TRUE borderline case...
Found 494 model-confirmed rejections out of 500 scanned.
BORDERLINE REJECTED APPLICANT (model-confirmed rejection)
  Monthly EMI   : ₹44,108
  Loan Amount   : ₹379,926
  Loan Tenure   : 5 years
  Interest Rate : 13.66%
  Credit Score  : 587.0
  DTI           : 46.87

Original approval probability: 0.3237
Original decision: REJECTED

GENERATING RECOMMENDATIONS

Top 10 configurations:
 Loan_Amount  Loan_Tenure  Approval_Prob  Amt_Change_%  Tenure_Change_Yr  Approved
      113977            7         0.4691         -70.0                 2     False
      113977           10         0.4682         -70.0                 5     False
      113977           25         0.4653         -70.0                20     False
      113977           20         0.4649         -70.0                15     False
      189963            7         0.4647         -50.0                 2     False
      151970            7         0.4638         -60.0   

In [6]:
# Multi-factor intervention: adjust loan + interest rate + collateral
print("=" * 60)
print("MULTI-FACTOR INTERVENTION TEST")
print("=" * 60)

recommendations_multi = []

for amt_factor in [0.7, 0.5, 0.4]:
    for tenure_add in [5, 10]:
        for rate_reduction in [0.9, 0.8, 0.7]:
            new_amt = int(sample['Loan_Amount'] * amt_factor)
            new_tenure = sample['Loan_Tenure'] + tenure_add
            new_rate = round(sample['Interest_Rate'] * rate_reduction, 2)
            candidate = build_features(
                sample,
                Loan_Amount=new_amt,
                Loan_Tenure=new_tenure,
                Interest_Rate=new_rate,
                Collateral='Yes',
                Property_Value=sample['Loan_Amount'] * 1.5,
                Loan_to_Value=70.0
            )
            prob = predict_probability(candidate)
            recommendations_multi.append({
                'Loan_Amount': new_amt,
                'Tenure': new_tenure,
                'Rate_%': new_rate,
                'Amt_Change_%': round((amt_factor - 1) * 100, 0),
                'Rate_Change_%': round((rate_reduction - 1) * 100, 0),
                'Approval_Prob': round(prob, 4),
                'Approved': prob >= 0.5
            })

multi_df = pd.DataFrame(recommendations_multi).sort_values('Approval_Prob', ascending=False)
print("\nTop 10 multi-factor configurations:")
print(multi_df.head(10).to_string(index=False))

flipped_multi = multi_df[multi_df['Approved'] == True]
print(f"\n{'=' * 60}")
print("MULTI-FACTOR CONFIGURATIONS THAT FLIP → APPROVAL")
print(f"{'=' * 60}")
if len(flipped_multi) > 0:
    print(flipped_multi.head(5).to_string(index=False))
else:
    print("Even multi-factor intervention cannot flip this applicant.")
    print("The applicant would need to significantly improve credit score")
    print("(6+ months of on-time payments) before reapplying.")

multi_df.to_csv(os.path.join(MODELS_DIR, 'recommendations_multi.csv'), index=False)
print(f"\n✅ Saved to models/recommendations_multi.csv")

MULTI-FACTOR INTERVENTION TEST

Top 10 multi-factor configurations:
 Loan_Amount  Tenure  Rate_%  Amt_Change_%  Rate_Change_%  Approval_Prob  Approved
      189963      10    9.56         -50.0          -30.0         0.7671      True
      189963      15    9.56         -50.0          -30.0         0.7615      True
      265948      10    9.56         -30.0          -30.0         0.7546      True
      151970      10    9.56         -60.0          -30.0         0.7540      True
      265948      15    9.56         -30.0          -30.0         0.7488      True
      151970      15    9.56         -60.0          -30.0         0.7481      True
      189963      10   10.93         -50.0          -20.0         0.7065      True
      189963      15   10.93         -50.0          -20.0         0.6996      True
      151970      10   10.93         -60.0          -20.0         0.6880      True
      265948      10   10.93         -30.0          -20.0         0.6861      True

MULTI-FACTOR CONFI